In [ ]:
import os
import numpy as np
import cv2
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

# ── 1. Load images ──────────────────────────────────────────
def load_fer2013(root, size=(48, 48)):
    X, y = [], []
    for emotion in os.listdir(root):
        folder = os.path.join(root, emotion)
        if not os.path.isdir(folder):
            continue
        for fname in os.listdir(folder):
            img = cv2.imread(os.path.join(folder, fname), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, size)
            X.append(img.flatten().astype(np.float64))
            y.append(emotion)
    return np.array(X), np.array(y)

X_train, y_train = load_fer2013("../datasets/fer2013/train")
X_test,  y_test  = load_fer2013("../datasets/fer2013/test")

# Normalise pixel values to [0, 1]
X_train /= 255.0
X_test  /= 255.0

# Encode string labels → integers
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test  = le.transform(y_test)

# ── 2. PCA (eigenfaces) ──────────────────────────────────────
n_components = 80  # try 50–150
pca = PCA(n_components=n_components, whiten=True)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

print(f"Explained variance: {pca.explained_variance_ratio_.sum():.1%}")

# ── 3. Train SVM classifier ──────────────────────────────────
clf = SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced')
clf.fit(X_train_pca, y_train)

# ── 4. Evaluate ──────────────────────────────────────────────
y_pred = clf.predict(X_test_pca)
print(classification_report(y_test, y_pred, target_names=le.classes_))

# ── 5. Predict a single image ────────────────────────────────
def predict_emotion(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (48, 48)).flatten().astype(np.float64) / 255.0
    weights = pca.transform(img.reshape(1, -1))
    label_idx = clf.predict(weights)[0]
    return le.inverse_transform([label_idx])[0]


Explained variance: 88.2%
              precision    recall  f1-score   support

       angry       0.34      0.38      0.36       958
     disgust       0.75      0.46      0.57       111
        fear       0.36      0.37      0.37      1024
       happy       0.61      0.64      0.62      1774
     neutral       0.45      0.45      0.45      1233
         sad       0.38      0.36      0.37      1247
    surprise       0.72      0.63      0.67       831

    accuracy                           0.48      7178
   macro avg       0.52      0.47      0.49      7178
weighted avg       0.49      0.48      0.48      7178



[ WARN:0@377.312] global loadsave.cpp:278 findDecoder imread_('my_face.jpg'): can't open/read file: check file path/integrity


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'resize'


In [ ]:
print(predict_emotion("../image copy.png"))

<>:1: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
<>:1: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
/tmp/ipykernel_36253/1091073377.py:1: SyntaxWarning: "\ " is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\ "? A raw string is also an option.
  print(predict_emotion("../image\ copy.png"))
[ WARN:0@739.856] global loadsave.cpp:278 findDecoder imread_('../image\ copy.png'): can't open/read file: check file path/integrity


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'resize'
